In [3]:
!pip install yt-dlp openai-whisper faiss-cpu sentence-transformers groq gradio
!apt-get install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 52.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 17.4 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=72d1c9ce939b3f9f3370f92ca486d1e46f627bc234a9d45a7bb75d6f77e8b5dd
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22

In [4]:
# import os
# os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"

import os

from google.colab import userdata

# Set your Groq API key
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [5]:
import yt_dlp
import os

def download_audio(url):
    # Clean old files
    for f in os.listdir():
        if f.startswith("audio"):
            os.remove(f)

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'audio.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
        }],
        'quiet': False
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return "audio.mp3"

In [6]:
import whisper

whisper_model = whisper.load_model("base")  # use "small" if better quality needed

def transcribe_audio(file_path):
    import os

    if not os.path.exists(file_path):
        return "❌ Audio file not found"

    if os.path.getsize(file_path) == 0:
        return "❌ Audio file is empty"

    result = whisper_model.transcribe(file_path)
    return result["text"]

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 119MiB/s]


In [7]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def create_vector_store(chunks):
    embeddings = embed_model.encode(chunks)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))

    return index, embeddings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
def retrieve(query, chunks, index, k=3):
    query_embedding = embed_model.encode([query])

    distances, indices = index.search(np.array(query_embedding), k)

    return [chunks[i] for i in indices[0]]

In [10]:
from groq import Groq

client = Groq()

def generate_answer(query, context):
    prompt = f"""
You are a helpful assistant.

Answer the question ONLY using the context below.
If the answer is not present, say: "Not found in video".

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # ✅ correct model
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [11]:
import os

def process_video(url):
    global chunks, index

    audio_path = download_audio(url)

    print("Downloaded file:", audio_path)

    if not os.path.exists(audio_path):
        return "❌ File does not exist"

    size = os.path.getsize(audio_path)
    print("File size:", size)

    if size == 0:
        return "❌ File is empty (download failed)"

    transcript = transcribe_audio(audio_path)

    if isinstance(transcript, str) and transcript.startswith("❌"):
        return transcript

    chunks = chunk_text(transcript)
    index, _ = create_vector_store(chunks)

    return "✅ Video processed successfully!"

In [12]:
def ask_question(query):
    global chunks, index

    if chunks is None:
        return "⚠️ Please load a video first."

    retrieved_chunks = retrieve(query, chunks, index)
    print("Retrieved chunks:", retrieved_chunks)

    if not retrieved_chunks:
        return "❌ No relevant context found"

    context = "\n\n".join(retrieved_chunks)

    return generate_answer(query, context)

In [ ]:
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown("# 🎥 YouTube RAG Q&A (Groq + Whisper)")

    with gr.Tab("Load Video"):
        url_input = gr.Textbox(label="YouTube URL")
        load_btn = gr.Button("Process Video")
        load_output = gr.Textbox()

    with gr.Tab("Ask Questions"):
        query_input = gr.Textbox(label="Ask something about the video")
        ask_btn = gr.Button("Get Answer")
        answer_output = gr.Textbox()

    load_btn.click(process_video, inputs=url_input, outputs=load_output)
    ask_btn.click(ask_question, inputs=query_input, outputs=answer_output)

app.launch(debug = True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1052d7803c0dd5fba8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[youtube] Extracting URL: https://www.youtube.com/watch?v=wYx9Q6dcJCw
[youtube] wYx9Q6dcJCw: Downloading webpage


[youtube] wYx9Q6dcJCw: Downloading android vr player API JSON
[info] wYx9Q6dcJCw: Downloading 1 format(s): 251
[download] Destination: audio.webm
[download] 100% of   34.56MiB in 00:00:01 at 27.70MiB/s  
[ExtractAudio] Destination: audio.mp3
Deleting original file audio.webm (pass -k to keep)
Downloaded file: audio.mp3
File size: 38017005
Retrieved chunks: ["If you're a software engineer or SaaS founder who loves writing code but dreads the marketing and sales grind, this video is for you. Simon, Severino, the CEO of Strategy Sprints, will show you how he practically automates his entire sales pipeline using artificial intelligence. Simon breaks down his exact system for using Claude, Obsidian, and Notion to create a virtual team of 45 collaborators that handles tedious admin tasks, lead generation, and cold outreach. You'll get a step-by-step demo on how to efficiently define your ideal client profile, seamlessly generate targeted lead lists using tools like Hunter and Craft AB tested